In [2]:
from torch.utils.data import Dataset
from tokenizers import Tokenizer

In [5]:
import torch

In [3]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        data,
        source_seq_len,
        target_seq_len,
        source_language: str,
        target_language: str,
        source_tokenizer: Tokenizer,
        target_tokenizer: Tokenizer
    ):
        super().__init__()
        self.data = data
        self.source_seq_len = source_seq_len
        self.target_seq_len = target_seq_len
        self.source_language = source_language
        self.target_language = target_language
        self.source_tokenizer = source_tokenizer
        self.target_tokenizer = target_tokenizer
        self.source_vocab_size = source_tokenizer.get_vocab_size()
        self.target_vocab_size = target_tokenizer.get_vocab_size()

    def __len__(self):
        return len(self.data)

    @staticmethod
    def create_token_and_mask(
        sentence: str,
        tokenizer: Tokenizer,
        seq_len: int,
        token_type: str
    ):
        mask = None
        token_ids = tokenizer.encode(sentence).ids

        sos_token_id_ls, eos_token_id_ls = [], []
        if token_type != "label":
            sos_token_id_ls = [tokenizer.token_to_id("<SOS>")]
        if token_type != "decoder":
            eos_token_id_ls = [tokenizer.token_to_id("<EOS>")]
        
        token_ids = sos_token_id_ls + token_ids + eos_token_id_ls
        if len(token_ids) > seq_len:
            raise ValueError

        pad_token_id = tokenizer.token_to_id("<PAD>")
        token_ids = torch.tensor(token_ids + [pad_token_id] * (seq_len - len(token_ids)), dtype=torch.int64)
        
        padding_mask = torch.ones(len(token_ids))
        padding_mask[token_ids == pad_token_id] = 0

        causal_mask = torch.tril(torch.ones(seq_len, seq_len))

        if token_type == "encoder":
            mask = padding_mask # 
        elif token_type == "decoder":
            # Pytorch broadcasts the padding mask across all rows of the causal_mask. Each row of causal_mask is multiplied by padding_mask
            mask = padding_mask.unsqueeze(0) * causal_mask # (1, seq_len) * (seq_len, seq_len)
        
        return token_ids, mask

    def __getitem__(self, idx):
        pair = self.data[idx]
        source_sentence = pair["translation"][self.source_language]
        source_token_ids, padding_mask = TranslationDataset.create_token_and_mask(
            sentence=source_sentence,
            tokenizer=self.source_tokenizer,
            seq_len=self.source_seq_len,
            token_type="encoder",
        )
        target_sentence = pair["translation"][self.target_language]
        target_token_ids, padding_causal_mask = TranslationDataset.create_token_and_mask(
            sentence=target_sentence,
            tokenizer=self.target_tokenizer,
            seq_len=self.target_seq_len,
            token_type="decoder",
        )
        label_token_ids, _ = TranslationDataset.create_token_and_mask(
            sentence=target_sentence,
            tokenizer=self.target_tokenizer,
            seq_len=self.target_seq_len,
            token_type="label",
        )
        
        data = {
            "source_sentence": source_sentence,
            "target_sentence": target_sentence,
            "encoder_input": source_token_ids,
            "decoder_input": target_token_ids,
            "label": label_token_ids,
            "encoder_mask": padding_mask.unsqueeze(0).unsqueeze(0),
            "decoder_mask": padding_causal_mask.unsqueeze(0),
        }

        return data

SyntaxError: expression expected after dictionary key and ':' (3303814922.py, line 22)